# 🎨 Тема 4. Раскрашиваем чёрно-белое видео (локальная версия)

План:
1. **Видео** — открываем, считаем кадры, берём один кадр.
2. **BGR → серый** — сами, по формуле.
3. **Серый → BGR** — пробуем вернуть цвет: сначала OpenCV, потом нейросетью. Получается плохо — разбираемся почему.
4. **Lab** — яркость L отдельно, цвет a, b отдельно. Сеть получает L и угадывает только a, b. Получается хорошо.
5. **Сохраняем** раскрашенное видео.

Каждое задание: замените `None` на свой код → **Shift + Enter** → запустите ячейку 🧪 под ним.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, видео и помощники { display-mode: "form" }
# Внутри: glue, show_video, read_video, sharpness, check. Читать не обязательно :)
import base64, os, shutil, subprocess
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import HTML, display

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(max(1, (os.cpu_count() or 2) // 2))


def glue(*imgs):
    # склеивает картинки в один ряд; серые делает трёхканальными
    prep = []
    for im in imgs:
        if im is None:
            raise ValueError("Показывать нечего: переменная пока None. Выполните задание")
        im = np.clip(np.asarray(im), 0, 255).astype(np.uint8)
        prep.append(cv2.cvtColor(im, cv2.COLOR_GRAY2BGR) if im.ndim == 2 else im)
    return np.hstack(prep)


def show_video(frames, fps=12, width=480):
    # показать список кадров как видео (нужен ffmpeg; без него — 6 кадров в ряд)
    if not frames or any(f is None for f in frames):
        print("⚠️ Показывать нечего: список пуст или в нём None")
        return
    frames = [cv2.cvtColor(f, cv2.COLOR_GRAY2BGR) if f.ndim == 2 else f for f in frames]
    if shutil.which("ffmpeg") is None:
        step = max(1, len(frames) // 6)
        show(*frames[::step][:6])
        return
    h, w = frames[0].shape[:2]
    h, w = h - h % 2, w - w % 2
    raw = b"".join(np.ascontiguousarray(f[:h, :w]).astype(np.uint8).tobytes() for f in frames)
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{w}x{h}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-f", "mp4", "-"]
    data = subprocess.run(cmd, input=raw, capture_output=True).stdout
    b64 = base64.b64encode(data).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))


def read_video(path, size=None, max_frames=1000):
    # прочитать видео в список кадров (size=(ширина, высота) — уменьшить)
    cap = cv2.VideoCapture(path)
    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.resize(frame, size) if size else frame)
    cap.release()
    return frames


def sharpness(img):
    # резкость: чем больше, тем чётче детали (дисперсия лапласиана)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    return cv2.Laplacian(g, cv2.CV_64F).var()


def _try_show(fn):
    # вызывает show() на красном квадрате и запоминает, что попало в plt.imshow
    got, orig = [], (plt.imshow, plt.show)
    plt.imshow = lambda a, *args, **kw: got.append(np.asarray(a))
    plt.show = lambda *args, **kw: None
    try:
        fn(np.full((4, 4, 3), (0, 0, 255), np.uint8))
    finally:
        plt.imshow, plt.show = orig
        plt.close("all")
    return got


def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True

os.system("wget -q -nc https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/Megamind.avi")
VIDEO = "Megamind.avi"
print("✅ Готово. Видео скачано:", os.path.exists(VIDEO))

---
# Разминка. Своя функция для показа картинок

Дальше мы будем часто смотреть на кадры. Напишем для этого функцию `show`: дали картинку — увидели её.

Готовая `glue(...)` склеивает несколько картинок в один ряд. Осталось нарисовать результат через matplotlib.
Подвох: OpenCV хранит цвета как **B, G, R**, а matplotlib ждёт **R, G, B**. Не переведёте — красное станет синим.

### ✏️ Задание 1. Допишите `show`

**Что должно получиться:** слева красный квадрат, справа синий.

In [ ]:
def show(*imgs, title=""):
    row = glue(*imgs)                     # готово: картинки в один ряд
    rgb = None   # ← из BGR в RGB
    plt.figure(figsize=(10, 4))
    # ← нарисуйте rgb: plt.imshow(...)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('картинка рисуется', lambda: len(_try_show(show)) == 1, 'Внутри нужен plt.imshow(rgb)'),
    ('красное осталось красным', lambda: tuple(_try_show(show)[0][0, 0]) == (255, 0, 0), 'cv2.cvtColor(row, cv2.COLOR_BGR2RGB)'),
], 1):
    show(np.full((60, 90, 3), (0, 0, 255), np.uint8), np.full((60, 90, 3), (255, 0, 0), np.uint8), title="красный | синий")

<details><summary>💡 Подсказка</summary>

Перевод цвета — `cv2.cvtColor(row, cv2.COLOR_BGR2RGB)`, рисует — `plt.imshow(rgb)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
rgb = cv2.cvtColor(row, cv2.COLOR_BGR2___)
plt.imshow(___)
```
</details>

---
# Часть 1. Видео

### ✏️ Задание 2. Откройте видео и узнайте число кадров `n_frames` и FPS `fps`

In [ ]:
cap = cv2.VideoCapture(VIDEO)
n_frames = None   # ← int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = None                  # ← cap.get(cv2.CAP_PROP_FPS)
cap.release()

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('кадров 270', lambda: n_frames == 270, 'int(cap.get(cv2.CAP_PROP_FRAME_COUNT))'),
    ('FPS около 24', lambda: fps is not None and 23 < fps < 25, 'cap.get(cv2.CAP_PROP_FPS)'),
], 2):
    print(f"Кадров: {n_frames}, FPS: {fps:.2f}, длительность: {n_frames / fps:.1f} с")

<details><summary>💡 Подсказка</summary>

Свойства видео читаются через `cap.get(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP____)
```
</details>

In [ ]:
all_frames = read_video(VIDEO, size=(360, 264))
print("Прочитано кадров:", len(all_frames), "| размер кадра:", all_frames[0].shape)
show_video(all_frames, fps=24)

### ✏️ Задание 3. Возьмите кадр номер 100: переменная `img`

In [ ]:
img = None

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('кадр взят', lambda: img is not None and img.shape == (264, 360, 3), 'all_frames[100]'),
    ('именно кадр 100', lambda: np.array_equal(img, all_frames[100]), 'Индекс 100'),
], 3):
    print("Пиксель в центре (B, G, R):", img[132, 180])
    show(img, title="кадр 100")

<details><summary>💡 Подсказка</summary>

`all_frames` — обычный список.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
img = all_frames[___]
```
</details>

---
# Часть 2. BGR → серый своими руками

У пикселя три числа B, G, R. В чёрно-белом — одно, яркость. Простейшая идея — **среднее**.

### ✏️ Задание 4. Серый кадр как среднее B, G, R: `gray_avg`

In [ ]:
gray_avg = None   # ← img.mean(axis=2).astype(np.uint8)

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('одно число на пиксель', lambda: gray_avg is not None and gray_avg.shape == (264, 360), 'axis=2 — ось каналов'),
    ('это среднее', lambda: np.abs(gray_avg.astype(int) - img.mean(axis=2).astype(int)).max() <= 1, 'img.mean(axis=2)'),
], 4):
    show(gray_avg, title="среднее B, G, R")

<details><summary>💡 Подсказка</summary>

`axis=2` — третья ось массива, каналы.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
gray_avg = img.mean(axis=___).astype(np.uint8)
```
</details>

Глаз чувствует цвета по-разному: зелёный кажется самым ярким, синий — самым тёмным. Поэтому OpenCV считает с весами:

$$серый = 0.114 \cdot B + 0.587 \cdot G + 0.299 \cdot R$$

### ✏️ Задание 5. Серый кадр по формуле с весами: `gray`

In [ ]:
B, G, R = cv2.split(img.astype(np.float32))
gray = None   # ← формула, потом .round().astype(np.uint8)

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('одно число на пиксель', lambda: gray is not None and gray.shape == (264, 360), 'Сложите каналы с весами'),
    ('совпадает с cv2.cvtColor', lambda: np.abs(gray.astype(int) - cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)).max() <= 1, '0.114 у B, 0.587 у G, 0.299 у R'),
], 5):
    show(gray_avg, gray, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), title="среднее | наша формула | cv2.COLOR_BGR2GRAY")
    print("Последние два совпадают")

<details><summary>💡 Подсказка</summary>

Каждый канал умножьте на свой вес и сложите.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
gray = (0.114 * B + 0.587 * ___ + 0.299 * ___).round().astype(np.uint8)
```
</details>

---
# Часть 3. Серый → BGR: пробуем вернуть цвет

### Попытка 1: OpenCV
Есть же `cv2.COLOR_GRAY2BGR`. Может, он вернёт цвет?

### ✏️ Задание 6. Переведите `gray` обратно в BGR: `back`

In [ ]:
back = None

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('три канала', lambda: back is not None and back.shape == (264, 360, 3), 'cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)'),
    ('цвета не появилось: B = G = R', lambda: np.array_equal(back[:, :, 0], back[:, :, 2]), ''),
], 6):
    show(img, back, title="оригинал img | back после GRAY2BGR")
    print("Пиксель в центре был", img[132, 180], "→ стал", back[132, 180])
    print("GRAY2BGR просто копирует серое число в три канала. Цвета нет.")

<details><summary>💡 Подсказка</summary>

Как обычный `cvtColor`, код `cv2.COLOR_GRAY2BGR`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
back = cv2.cvtColor(gray, cv2.COLOR_GRAY2___)
```
</details>

### Почему формулой не вернуть

### ✏️ Задание 7. Посчитайте серый для красного (B=0, G=0, R=255) и зелёного (B=0, G=130, R=0)

In [ ]:
gray_red = None     # подставьте в формулу
gray_green = None

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('красный', lambda: gray_red is not None and abs(gray_red - 76.2) < 1, '0.299 * 255'),
    ('зелёный', lambda: gray_green is not None and abs(gray_green - 76.3) < 1, '0.587 * 130'),
], 7):
    print(f"красный → {gray_red:.0f}, зелёный → {gray_green:.0f}")
    show(np.full((80, 120, 3), (0, 0, 255), np.uint8), np.full((80, 120, 3), (0, 130, 0), np.uint8), np.full((80, 120, 3), 76, np.uint8), title="красный | зелёный | серый 76")
    print("Два разных цвета → один серый 76. По 76 обратно не понять, что было.")

<details><summary>💡 Подсказка</summary>

Подставьте B, G, R в формулу.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
gray_red = 0.299 * ___
gray_green = 0.587 * ___
```
</details>

### Попытка 2: нейросеть угадывает B, G, R по серому
Формулы нет — пусть угадывает нейросеть по **смыслу** («это лицо — оно телесное»). Учим на кадрах самого мультфильма: вход — серый кадр, ответ — цветной.

Кадры уменьшаем до 128×96, чтобы обучение шло ~20 секунд. Первые 200 — обучение, последние 70 — проверка (сеть их не видит).

In [ ]:
frames = read_video(VIDEO, size=(128, 96))
train_frames, test_frames = frames[:200], frames[200:]


def augment(frame):
    # случайные поворот, масштаб, сдвиг и зеркало — чтобы сеть не зазубривала кадры
    h, w = frame.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), np.random.uniform(-10, 10), np.random.uniform(0.9, 1.15))
    M[:, 2] += np.random.uniform(-8, 8, 2)
    out = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    return cv2.flip(out, 1) if np.random.rand() < 0.5 else out


def make_net(out_channels):
    # автоэнкодер: энкодер сжимает 96×128 → 12×16, декодер разворачивает обратно
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
        nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
        nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
        nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
        nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
        nn.ConvTranspose2d(32, out_channels, 4, stride=2, padding=1), nn.Sigmoid())   # выход 0…1


def bgr_tensors(imgs):
    # вход: серый кадр (N, 1, H, W); ответ: цветной (N, 3, H, W); всё делим на 255 → 0…1
    g = np.stack([cv2.cvtColor(i, cv2.COLOR_BGR2GRAY) for i in imgs])[:, None] / 255
    c = np.stack(imgs).transpose(0, 3, 1, 2) / 255
    return torch.tensor(g, dtype=torch.float32), torch.tensor(c, dtype=torch.float32)


x, y = bgr_tensors(test_frames)
print("вход:", tuple(x.shape), " ответ:", tuple(y.shape))

### Шаг обучения — 4 действия
1. `pred = model(x)` — сеть отвечает;
2. `loss = loss_fn(pred, y)` — насколько ошиблась (средний квадрат ошибки);
3. `opt.zero_grad()` и `loss.backward()` — считаем, куда двигать каждый вес;
4. `opt.step()` — двигаем веса.

Функция `train_step` пригодится дважды: и для BGR, и для Lab.

### ✏️ Задание 8. Допишите `train_step`

In [ ]:
loss_fn = nn.MSELoss()

def train_step(model, opt, x, y):
    # 1) pred = ...   2) loss = ...   3) opt.zero_grad() и loss.backward()   4) opt.step()
    loss = None
    return loss.item()

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('веса меняются', lambda: (lambda m: (lambda w0: (train_step(m, torch.optim.SGD(m.parameters(), 0.1), torch.ones(4, 1), torch.zeros(4, 1)), m.weight.item() != w0)[1])(m.weight.item()))(nn.Linear(1, 1)), 'Без opt.step() веса не меняются'),
    ('возвращает ошибку числом', lambda: isinstance(train_step(nn.Linear(1, 1), torch.optim.SGD(nn.Linear(1, 1).parameters(), 0.1), torch.ones(4, 1), torch.zeros(4, 1)), float), 'return loss.item()'),
], 8):
    pass

<details><summary>💡 Подсказка</summary>

Четыре строчки по порядку: forward → loss → zero_grad + backward → step.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
pred = model(x)
loss = loss_fn(pred, ___)
opt.zero_grad()
loss.___()
opt.___()
```
</details>

In [ ]:
def train(model, to_tensors, epochs=20, use_augment=True):
    # цикл обучения: эпохи → пачки по 16 кадров → train_step
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    x_test, y_test = to_tensors(test_frames)
    for epoch in range(epochs):
        order = np.random.permutation(len(train_frames))
        for i in range(0, len(order), 16):
            batch = [train_frames[j] for j in order[i:i + 16]]
            if use_augment:
                batch = [augment(f) for f in batch]
            train_step(model, opt, *to_tensors(batch))
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                print(f"эпоха {epoch + 1:2d}: ошибка на проверке {loss_fn(model(x_test), y_test).item():.4f}")
    return model


torch.manual_seed(0); np.random.seed(0)
net_bgr = train(make_net(3), bgr_tensors)     # 3 выхода: B, G, R
with torch.no_grad():
    pred = net_bgr(bgr_tensors(test_frames)[0]).numpy()
colored_bgr = [(p.transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8) for p in pred]

k = 30
gray3 = cv2.cvtColor(cv2.cvtColor(test_frames[k], cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
show(gray3, colored_bgr[k], test_frames[k], title="серый вход | сеть угадала BGR | оригинал")
print(f"резкость: оригинал {sharpness(test_frames[k]):.0f}, сеть BGR {sharpness(colored_bgr[k]):.0f}")

### ❓ Что пошло не так
Цвет примерно угадан, но картинка **мутная**: пропали очки, глаза, волосы.

Сеть должна выдать **все три числа** B, G, R — а в каждом из них спрятана **яркость**. Значит, она заново рисует и яркость, и все детали, хотя они у нас **уже есть** в сером кадре. Сжатие в автоэнкодере (96×128 → 12×16) мелкие детали теряет.

**Хочется:** яркость взять готовую из серого кадра, а сеть пусть угадывает **только цвет**. Нужен формат, где яркость — отдельный канал. Это **Lab**.

---
# Часть 4. Lab: яркость отдельно, цвет отдельно

* **L** — яркость (≈ серый кадр);
* **a** — ось «зелёный ↔ красный»;
* **b** — ось «синий ↔ жёлтый».

В OpenCV все от 0 до 255, **128 у a и b — «цвета нет»**.

### ✏️ Задание 9. Переведите `img` в Lab и разложите на каналы `L`, `a`, `b`

**Что должно получиться:** L очень похож на `gray` из части 2 — это та же яркость.

In [ ]:
lab = None
L, a, b = None, None, None

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('перевод в Lab', lambda: lab is not None and lab.shape == img.shape and not np.array_equal(lab, img), 'cv2.cvtColor(img, cv2.COLOR_BGR2LAB)'),
    ('каналы', lambda: L is not None and np.array_equal(L, lab[:, :, 0]), 'cv2.split(lab)'),
], 9):
    show(L, a, b, title="L | a | b")
    print("L — обычное ч/б фото; a и b — почти серые, пятна там, где есть цвет")
    print("Разница L и нашего gray (в среднем):", round(np.abs(L.astype(int) - gray).mean(), 1))

<details><summary>💡 Подсказка</summary>

`cv2.COLOR_BGR2LAB`, потом `cv2.split`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
lab = cv2.cvtColor(img, cv2.COLOR_BGR2___)
L, a, b = cv2.split(___)
```
</details>

### ✏️ Задание 10. Сотрите цвет: a и b → 128, соберите и переведите в BGR (`no_color`)

In [ ]:
flat = np.full_like(L, 128)
no_color = None   # ← merge([L, flat, flat]) → LAB2BGR

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
if check([
    ('кадр собран', lambda: no_color is not None and no_color.shape == img.shape, 'cv2.cvtColor(cv2.merge([L, flat, flat]), cv2.COLOR_LAB2BGR)'),
    ('цвета нет', lambda: np.abs(no_color[:, :, 0].astype(int) - no_color[:, :, 2]).max() <= 3, 'a и b — flat'),
], 10):
    show(img, no_color, title="оригинал | L + «нет цвета»")
    print("L + «нет цвета» = чёрно-белый кадр. Сеть будет делать обратное: по L угадывать a и b.")

<details><summary>💡 Подсказка</summary>

`cv2.merge` склеивает каналы, `cv2.COLOR_LAB2BGR` — обратно в BGR.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
no_color = cv2.cvtColor(cv2.merge([L, flat, ___]), cv2.COLOR_LAB2___)
```
</details>

### Попытка 3: нейросеть угадывает только a и b
Вход — L, ответ — a и b. Допишите функцию, которая готовит данные (как `bgr_tensors`, но из Lab).

### ✏️ Задание 11. Допишите `lab_tensors`: вход L, ответ ab (всё делим на 255)

In [ ]:
def lab_tensors(imgs):
    lab = np.stack([cv2.cvtColor(i, cv2.COLOR_BGR2LAB) for i in imgs]).transpose(0, 3, 1, 2) / 255   # (N, 3, H, W)
    L = lab[:, :1]                       # канал 0 — вход
    ab = None                          # ← каналы 1 и 2 — ответ: lab[:, 1:]
    return torch.tensor(L, dtype=torch.float32), torch.tensor(ab, dtype=torch.float32)

In [ ]:
#@title 🧪 Проверка задания 11 { display-mode: "form" }
if check([
    ('вход — 1 канал', lambda: tuple(lab_tensors(test_frames[:2])[0].shape) == (2, 1, 96, 128), 'L = lab[:, :1]'),
    ('ответ — 2 канала', lambda: tuple(lab_tensors(test_frames[:2])[1].shape) == (2, 2, 96, 128), 'ab = lab[:, 1:]'),
], 11):
    print("L:", tuple(lab_tensors(test_frames)[0].shape), " ab:", tuple(lab_tensors(test_frames)[1].shape))

<details><summary>💡 Подсказка</summary>

Каналы — ось 1. Нужны все, начиная с первого: `1:`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
ab = lab[:, ___:]
```
</details>

In [ ]:
torch.manual_seed(0); np.random.seed(0)
net_lab = train(make_net(2), lab_tensors)     # 2 выхода: a, b

### ✏️ Задание 12. Раскрасьте проверочные кадры: настоящий L + угаданные ab → BGR (`colored_lab`)

**Что должно получиться:** Lab-версия такая же чёткая, как оригинал: яркость настоящая, сеть добавила только цвет.

In [ ]:
L_test, _ = lab_tensors(test_frames)
with torch.no_grad():
    ab_pred = net_lab(L_test)
lab_pred = torch.cat([L_test, ab_pred], dim=1).numpy()            # склеили L и ab: (N, 3, H, W)
lab_pred = (lab_pred.transpose(0, 2, 3, 1) * 255).clip(0, 255).astype(np.uint8)
colored_lab = [None for x in lab_pred]   # ← каждый x из Lab в BGR

In [ ]:
#@title 🧪 Проверка задания 12 { display-mode: "form" }
if check([
    ('кадров 70', lambda: len(colored_lab) == 70, 'Список по всем x из lab_pred'),
    ('цветные кадры', lambda: colored_lab[0] is not None and colored_lab[0].shape == (96, 128, 3) and np.abs(colored_lab[30][:, :, 0].astype(int) - colored_lab[30][:, :, 2]).mean() > 5, 'cv2.cvtColor(x, cv2.COLOR_LAB2BGR)'),
], 12):
    show(gray3, colored_bgr[k], colored_lab[k], test_frames[k], title="серый | сеть BGR | сеть Lab | оригинал")
    print(f"резкость: оригинал {sharpness(test_frames[k]):.0f}, BGR {sharpness(colored_bgr[k]):.0f}, Lab {sharpness(colored_lab[k]):.0f}")
    err = lambda imgs: np.mean([np.abs(p.astype(int) - t).mean() for p, t in zip(imgs, test_frames)])
    print(f"средняя ошибка цвета: BGR {err(colored_bgr):.1f}, Lab {err(colored_lab):.1f}")

<details><summary>💡 Подсказка</summary>

Обратный перевод — `cv2.COLOR_LAB2BGR`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
colored_lab = [cv2.cvtColor(x, cv2.COLOR_LAB2___) for x in lab_pred]
```
</details>

### ✅ Вывод
| | BGR | Lab |
|---|---|---|
| сеть угадывает | 3 числа: B, G, R | 2 числа: a, b |
| яркость | рисует заново → мутно | берём настоящую из серого |
| детали | теряются | сохраняются |

Синий свитер всё равно бурый: редкий цвет, и средний квадрат ошибки тянет ответ к «среднему» цвету.

In [ ]:
gray_test = [cv2.cvtColor(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR) for f in test_frames]
show_video([np.hstack([g, cb, cl, f]) for g, cb, cl, f in zip(gray_test, colored_bgr, colored_lab, test_frames)], width=720)
print("серый | сеть BGR | сеть Lab | оригинал")

---
# Часть 5. Сохраняем видео

### ✏️ Задание 13. Сохраните `colored_lab` в `video/colorized.mp4` (FPS 12, размер `(128, 96)`)

In [ ]:
os.makedirs("video", exist_ok=True)
out = None
# цикл out.write(...) и out.release()

In [ ]:
#@title 🧪 Проверка задания 13 { display-mode: "form" }
if check([
    ('файл создан', lambda: os.path.exists('video/colorized.mp4') and os.path.getsize('video/colorized.mp4') > 1000, 'VideoWriter, цикл write, release'),
    ('все кадры записаны', lambda: int(cv2.VideoCapture('video/colorized.mp4').get(cv2.CAP_PROP_FRAME_COUNT)) == 70, 'Размер — (128, 96): (ширина, высота)'),
], 13):
    print("video/colorized.mp4:", os.path.getsize("video/colorized.mp4") // 1024, "КБ")

<details><summary>💡 Подсказка</summary>

`cv2.VideoWriter(имя, fourcc, fps, (ширина, высота))`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
out = cv2.VideoWriter("video/colorized.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 12, (128, 96))
for f in colored_lab:
    out.___(f)
out.___()
```
</details>

### 🔬 Эксперименты
* `train(make_net(2), lab_tensors, use_augment=False)` — ошибка на проверке без аугментации.
* `epochs=40` — станет ли лучше?
* Поменяйте `k = 30` на другой кадр.